# AI-Augmented Documentation Pipeline
## From static docs site to quality-enforced, agent-ready corpus

**Author:** Tom Aciukewicz  
**Project:** [tazdocs-as-code](https://taz-mon.github.io/tazdocs-as-code)  
**Tools:** Anthropic Python SDK · VectorLint · afdocs

---

This notebook walks through the core technical implementation behind a four-phase documentation quality pipeline built on my Docusaurus portfolio site. The goal: enforce consistent style and tone through LLM-as-a-judge evaluation, then validate the site for agent-friendly consumption.

It demonstrates three things:

1. **Style enforcement without drift:** uses a `VECTORLINT.md` system prompt, which is a rule-based evaluation to catch prose problems like a skilled editor would. It is not a syntax linter
2. **Intelligent prose review:** calls the Anthropic API with a structured judge rule, parsing the response, and interpreting weighted scores
3. **Agent-readiness validation:** checks whether a documentation site is consumable by coding agents, not just human readers

You can run this against your own documentation. Swap in any Markdown or MDX file where indicated.

## Setup

Install the Anthropic Python SDK. This notebook uses `claude-sonnet-4-6`, which strikes a balance of evaluation quality and latency for CI use. Update the model string in section 5 to align with your organization's version requirements.

```bash
pip install anthropic
```

Set your API key as an environment variable before running:

```bash
export ANTHROPIC_API_KEY="your-key-here"
```

In [ ]:
import os
import json
import anthropic

api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY environment variable is not set. "
        "Run: export ANTHROPIC_API_KEY='your-key-here'"
    )

client = anthropic.Anthropic(api_key=api_key)

print(f"SDK version: {anthropic.__version__}")
print("Client initialized.")

SDK version: 0.89.0
Client initialized.


## Part 1 — The style contract: `VECTORLINT.md`

The key to consistent AI-assisted drafting is a global style contract that prepends to every evaluation. Without it, each API call starts from scratch and voice drifts across the corpus.

`VECTORLINT.md` lives at the project root. VectorLint prepends its contents to the system prompt for every rule it runs. In a standalone API call like this notebook, we load it manually and pass it as the system message.

The rules below are a subset of the full style guide used in `tazdocs-as-code`. They encode the decisions that matter most for consistency: voice, tense, prohibited constructions, and terminology discipline.

In [2]:
# In production, load this from the file:
#   with open("VECTORLINT.md", "r") as f:
#       VECTORLINT_MD = f.read()
#
# For this notebook, the content is inline so it runs without the repo.

VECTORLINT_MD = """
## Voice and tone

- Write in active voice. Never use passive constructions.
- Use present tense throughout.
- Address the reader as "you." Do not use "the user," "the developer," or "one."
- Do not call any step or feature "simple," "easy," "straightforward," or "just."
- Do not use marketing language, superlatives, or exclamation marks.
- Do not use filler phrases: "please note," "at this time," "it is worth mentioning."
- Do not use weasel words when the correct answer is known: "might," "could," "generally," "typically."
- Do not attribute motivations or feelings to software. Systems execute; they do not think, want, or know.
- Contractions are acceptable and preferred when they sound natural.

## Structure

- Every page must open with a short description (1-2 sentences) stating what the page covers and who it is for.
- Use sentence case for all headings. Capitalize only the first word and proper nouns.
- Do not end headings with punctuation.
- Do not stack two headings with no body content between them.

## Language

- Define every acronym on first use: full term, then acronym in parentheses.
- After first use, use the acronym consistently. Do not alternate.
- Do not use "e.g.," "i.e.," or "etc." Use "for example," "that is," and "and so on."
- Use one term per concept. Do not introduce synonyms to avoid repetition.
"""

print(f"Style contract loaded: {len(VECTORLINT_MD.split())} words")

Style contract loaded: 228 words


## Part 2 — The sample document

This is the real `docs/intro.md` from `tazdocs-as-code` — the introductory page of the portfolio site. It's a genuine first-generation page written before the style pipeline existed, which makes it an honest test subject.

The evaluator will surface real issues in this content, including:

- Informal register that drifts from the style guide voice (`I call it`, `off my friends and family`)
- `IDE` defined on first use but `Git` used undefined
- Weasel words (`seems important`, `I believe`)
- A typo in a product name (`Docusaraus` instead of `Docusaurus`)
- Passive constructions and hedged claims

This is exactly the kind of content the pipeline is designed to catch before it reaches a human reviewer — or a RAG system.

In [3]:
# In production, load the file directly from your repo:
#   with open("docs/intro.md", "r") as f:
#       SAMPLE_DOC = f.read()
#
# The content below is the real docs/intro.md from tazdocs-as-code.

SAMPLE_DOC = """
---
sidebar_position: 1
---

# Why docs-as-code?

Documentation-as-code is a process writers and developers use to make changes locally and test them before pushing
them to a live documentation site. To get started, you need an Integrated Development Environment (IDE), such as
Visual Code Studio, a static site generator, a Git repository, a change management system like Github to set up
workflows, and a hosting platform.

## Why do I, a technical writer, use docs-as-code?

To impress my friends and family by typing something in a window of my computer and showing off how it makes the
letters appear like magic on the internet. I call it \"Command-line and Teller\".

## Use the site as my resume

I created this site using Docusaraus, by following this great tutorial, How to Set Up Documentation as Code with
Docusaurus and GitHub Actions. Follow it to learn about this and related tools which technical writers can explore
to learn how to adopt to today's technical writing economy. The field is changing and being familiar with static
doc site generators seems important. I believe it brings up your tech tools game.

You may not be using it at your current job, but if you want to find newer tech writing roles, you should be on
the lookout for challenges like this. Trying these tools shows that you aren't sitting back, relying on old
technology to document today's technology. This is one step in showing you that I see how the landscape is
changing, and I am trying to change with it.

## Linux computers

This is a great place to get started with a native Linux computer. Something to think about. In a day or two
you'll wonder why you hesitated.
"""

print(f"Document loaded: {len(SAMPLE_DOC.split())} words")
print("\nFirst 200 characters:")
print(SAMPLE_DOC[:200])

Document loaded: 287 words

First 200 characters:

---
sidebar_position: 1
---

# Why docs-as-code?

Documentation-as-code is a process writers and developers use to make changes locally and test them before pushing
them to a live documentation site.


## Part 3 — The judge rule

VectorLint supports two evaluation modes. This notebook uses a **judge rule** — the model scores content across weighted criteria using a 1–4 rubric, and VectorLint normalizes each score to a 1–10 scale.

The criteria and their weights encode what matters most. `VoiceAndTone` carries the highest weight here (40) because drift in voice is the hardest problem to catch with deterministic linters and the most damaging to corpus consistency.

The prompt instructs the model to return structured JSON — one score per criterion with a brief rationale. Structured output is essential for CI: you need a number you can threshold, not a paragraph you have to parse.

In [4]:
JUDGE_RULE_CRITERIA = [
    {"name": "Voice and tone",     "id": "VoiceAndTone",     "weight": 40},
    {"name": "Structure",          "id": "Structure",         "weight": 30},
    {"name": "Language precision", "id": "LanguagePrecision", "weight": 30},
]

JUDGE_RULE_PROMPT = """
You are a documentation quality evaluator. Score the content below against the style rules in your system prompt.

Score each criterion using this rubric:
  4 = Excellent — fully compliant, no issues detected
  3 = Good — minor issues, does not significantly affect quality
  2 = Fair — several issues present, noticeable quality impact
  1 = Poor — pervasive issues, does not meet the standard

Return ONLY valid JSON in this exact format — no preamble, no markdown fences:
{
  "scores": [
    {"id": "VoiceAndTone",     "score": <1-4>, "rationale": "<one sentence>"},
    {"id": "Structure",         "score": <1-4>, "rationale": "<one sentence>"},
    {"id": "LanguagePrecision", "score": <1-4>, "rationale": "<one sentence>"}
  ]
}
"""

print("Judge rule configured.")
print(f"Criteria: {[c['name'] for c in JUDGE_RULE_CRITERIA]}")
print(f"Weights:  {[c['weight'] for c in JUDGE_RULE_CRITERIA]}")

Judge rule configured.
Criteria: ['Voice and tone', 'Structure', 'Language precision']
Weights:  [40, 30, 30]


## Part 4 — Call the Anthropic API

The system message is the `VECTORLINT.md` style contract combined with the judge rule prompt. The user message is the document content.

This mirrors exactly how VectorLint constructs its API call — the style contract sets the baseline context, and the rule prompt defines the evaluation task on top of it.

In [1]:
system_prompt = VECTORLINT_MD.strip() + "\n\n" + JUDGE_RULE_PROMPT.strip()

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_completion_tokens=1024,
    system=system_prompt,
    messages=[
        {
            "role": "user",
            "content": f"Evaluate this documentation content:\n\n{SAMPLE_DOC}"
        }
    ]
)

raw_response = message.content[0].text
print("Raw API response:")
print(raw_response)

NameError: name 'VECTORLINT' is not defined

## Part 5 — Parse and score

The raw response is structured JSON. We parse it, look up the weight for each criterion, and compute a weighted average normalized to a 1–10 scale.

The normalization maps the 1–4 rubric to 1–10 using the VectorLint convention: `4 → 10.0`, `3 → 7.0`, `2 → 4.0`, `1 → 1.0`. The weighted average then determines whether the document passes the quality gate.

In CI, the pipeline compares this score against a configured threshold. Scores below 5 block merge; scores between 5 and 7 post a warning comment.

In [ ]:
SCORE_MAP = {4: 10.0, 3: 7.0, 2: 4.0, 1: 1.0}
WEIGHT_MAP = {c["id"]: c["weight"] for c in JUDGE_RULE_CRITERIA}

clean = raw_response.strip().removeprefix("```json").removesuffix("```").strip()
result = json.loads(clean)
scores = result["scores"]

total_weight = sum(WEIGHT_MAP.values())
weighted_sum = 0.0

print(f"{'Criterion':<22} {'Raw':>5} {'Normalized':>11} {'Weight':>7}")
print("-" * 50)

for entry in scores:
    cid        = entry["id"]
    raw_score  = entry["score"]
    normalized = SCORE_MAP[raw_score]
    weight     = WEIGHT_MAP[cid]
    weighted_sum += normalized * weight
    print(f"{cid:<22} {raw_score:>5} {normalized:>11.1f} {weight:>7}")

final_score = weighted_sum / total_weight
print("-" * 50)
print(f"{'Weighted final score':<22} {final_score:>17.2f}")

## Part 6 — Apply the quality gate

The pipeline responds to scores according to configured thresholds. These match the operating model in the `tazdocs-as-code` implementation:

| Score | Meaning | Pipeline behavior |
|-------|---------|-------------------|
| 9–10  | Clean | Pass silently |
| 7–8   | Minor issues | Warning comment, allow merge |
| 5–6   | Needs work | Warning comment, allow merge |
| < 5   | Failing | Error; blocks merge if severity = error |

In [ ]:
def apply_quality_gate(score):
    if score >= 9:
        return "PASS",    "Clean — no action required."
    elif score >= 7:
        return "WARN",    "Minor issues detected. Review feedback before merging."
    elif score >= 5:
        return "WARN",    "Content needs work. Address feedback before merging."
    else:
        return "FAIL",    "Quality gate failed. Merge is blocked until issues are resolved."

status, message = apply_quality_gate(final_score)

print(f"\nFinal score:  {final_score:.2f} / 10")
print(f"Gate status:  {status}")
print(f"Message:      {message}")

print("\n--- Criterion rationales ---")
for entry in scores:
    print(f"\n{entry['id']} (score: {entry['score']}/4)")
    print(f"  {entry['rationale']}")

## Part 7 — Agent-readiness check

Style quality is one dimension. The other is whether the documentation site is usable by AI agents — not just human readers.

[Dachary Carey's research](https://agentdocsspec.com) into how coding agents consume documentation found that agents don't browse — they fetch specific URLs from memory, fail silently when those URLs don't resolve, and rarely try to recover. A site built for human navigation is a site agents use badly.

The [`afdocs` CLI](https://github.com/agent-ecosystem/afdocs) implements the Agent-Friendly Documentation Spec: 22 checks across 7 categories. The checks below replicate the most critical ones you can validate locally before running the full CLI against a deployed site.

In the `tazdocs-as-code` pipeline, `afdocs` runs as a post-deployment gate on every push to `main`.

In [ ]:
import urllib.request
import urllib.error

# Replace with your own site URL to run against your deployment.
SITE_URL = "https://taz-mon.github.io/tazdocs-as-code"

def check_url(url, label):
    """Fetch a URL and return (status_code, ok, label)."""
    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "afdocs-notebook-check/1.0"}
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status, True, label
    except urllib.error.HTTPError as e:
        return e.code, False, label
    except Exception as e:
        return 0, False, f"{label} ({type(e).__name__})"

checks = [
    (f"{SITE_URL}/llms.txt",       "llms.txt index present"),
    (f"{SITE_URL}/docs/intro",     "Docs root resolves without trailing slash"),
    (f"{SITE_URL}/docs/intro.md",  "Markdown version available at .md path"),
    (f"{SITE_URL}/sitemap.xml",    "sitemap.xml present for agent discovery"),
]

print(f"Agent-readiness checks for: {SITE_URL}\n")
print(f"{'Check':<45} {'Status':>8} {'Result':>8}")
print("-" * 65)

results = []
for url, label in checks:
    code, ok, lbl = check_url(url, label)
    symbol = "PASS" if ok else "FAIL"
    print(f"{lbl:<45} {code:>8} {symbol:>8}")
    results.append(ok)

passed = sum(results)
total  = len(results)
print("-" * 65)
print(f"\nAgent-readiness: {passed}/{total} checks passed")

if passed < total:
    print("\nNext step: run the full afdocs CLI for detailed remediation guidance.")
    print("  npx afdocs check --url", SITE_URL)

## What this pipeline demonstrates

This notebook implements three things that matter for a production documentation pipeline:

**Style consistency without drift.** The `VECTORLINT.md` system prompt acts as a global style contract. Every evaluation call starts from the same baseline. Voice, tense, prohibited constructions, and terminology rules apply automatically — without a human re-reading the style guide for each review.

**Editorial review, not spell-checking.** The judge rule evaluates content the way a skilled editor reads it — looking for passive constructions, weasel words, and structural problems that no regex will find. The rationale fields explain *what* the issue is, so the author can act on the feedback rather than guess at it.

**Agent-readiness as a first-class concern.** Style quality and human readability are necessary but not sufficient. Coding agents, RAG systems, and AI tools increasingly consume documentation as structured input — and most documentation sites weren't built for that access pattern. The agent-readiness checks make this a measurable property, not an afterthought.

---

In the full `tazdocs-as-code` pipeline, this evaluation logic runs as a GitHub Actions workflow on every pull request — path-scoped to documentation files, capped at 20 files per run to keep CI fast, and configured with per-pattern strictness levels in `.vectorlint.ini`.

**Links:**
- Portfolio: [taz-mon.github.io/tazdocs-as-code](https://taz-mon.github.io/tazdocs-as-code)
- VectorLint docs: [vectorlint.mintlify.app](https://vectorlint.mintlify.app)
- Agent-Friendly Documentation Spec: [agentdocsspec.com](https://agentdocsspec.com)
- `afdocs` CLI: [github.com/agent-ecosystem/afdocs](https://github.com/agent-ecosystem/afdocs)